Importamos las librerias de Spark

In [0]:
!pip install pyspark

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

Paso 1. Cargar y asignar nombres a las columnas

Creamos la sesion

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date

spark = SparkSession.builder.appName("Villarica").getOrCreate()

Cargamos el archivo de datos csv de Villarica

In [0]:
df = spark.read.csv("/Volumes/workspace/bd_villarica/grupo_01/n00428590/RESULTADOS-Villarica.csv",header=False,sep=";",inferSchema=True,)

Damos Nombres a lasa columnas

In [0]:
df = df.toDF(
    "id_fact_venta",
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "metodo_pago"
)

Paso 2. Revisar la estructura de los datos

In [0]:
# Mostrar tipos de datos
df.printSchema()

# Mostrar estadísticas generales
df.describe().show()

# Contar registros
print("Total de registros:", df.count())

root
 |-- id_fact_venta: integer (nullable = true)
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)
 |-- metodo_pago: string (nullable = true)

+-------+----------------+-----------------+--------------------+----------------+------------------+-----------------+-----------+
|summary|   id_fact_venta|         id_venta|            producto|       categoria|          cantidad|           precio|metodo_pago|
+-------+----------------+-----------------+--------------------+----------------+------------------+-----------------+-----------+
|  count|            2197|             2197|                2197|            2197|              2197|             2197|       2197|
|   mean|          1099.0|632.1993627674101|                NULL|            NULL|2.3245334547109695|9.505962676376871|       NULL|
| stddev|

Paso 3. Eliminar espacios innecesarios

In [0]:
# Limpiar espacios en columnas de texto
df = df.withColumn("producto", trim(col("producto"))).withColumn("categoria", trim(col("categoria"))).withColumn("metodo_pago", trim(col("metodo_pago")))

df.show(5, truncate=False)

+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
|id_fact_venta|id_venta|fecha     |producto          |categoria        |cantidad|precio|metodo_pago|
+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
|1            |1       |2026-10-01|Torta de chocolate|Pasteles y tortas|1       |7.0   |Efectivo   |
|2            |1       |2026-10-01|Keke de naranja   |Pasteles y tortas|1       |4.0   |Efectivo   |
|3            |2       |2026-10-01|Té verde          |Tés e infusiones |4       |14.0  |Efectivo   |
|4            |3       |2026-10-01|Café americano    |Cafés            |6       |24.0  |Efectivo   |
|5            |4       |2026-10-01|Té verde          |Tés e infusiones |2       |7.0   |Yape       |
+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
only showing top 5 rows


Paso 4. Identificar y tratar valores nulos

In [0]:
# Revisar valores nulos
df.select([col(c).isNull().cast("int").alias(c)
    for c in df.columns
]).groupBy().sum().show()

df = df.na.drop(
    subset=[
        "id_fact_venta",
        "id_venta",
        "fecha",
        "producto",
        "categoria",
        "cantidad",
        "precio",
        "metodo_pago"
    ]
)

+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|sum(id_fact_venta)|sum(id_venta)|sum(fecha)|sum(producto)|sum(categoria)|sum(cantidad)|sum(precio)|sum(metodo_pago)|
+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|                 0|            0|         0|            0|             0|            0|          0|               0|
+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+



Paso 5. Corregir tipos de datos

In [0]:
df = df.withColumn("id_fact_venta", col("id_fact_venta").cast("integer")).withColumn("id_venta", col("id_venta").cast("integer")).withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd")).withColumn("cantidad", col("cantidad").cast("integer")).withColumn("precio", col("precio").cast("decimal(12,2)"))

df.printSchema()

root
 |-- id_fact_venta: integer (nullable = true)
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: decimal(12,2) (nullable = true)
 |-- metodo_pago: string (nullable = true)



Paso 6. Filtrar registros incorrectos

In [0]:
df = df.filter(
    (col("id_fact_venta") > 0) &
    (col("id_venta") > 0) &
    (col("fecha").isNotNull()) &
    (col("producto") != "") &
    (col("categoria") != "") &
    (col("cantidad") > 0) &
    (col("precio") > 0) &
    (col("metodo_pago") != "")
)

df.show(10, truncate=False)

+-------------+--------+----------+--------------------------------+-----------------+--------+------+-----------+
|id_fact_venta|id_venta|fecha     |producto                        |categoria        |cantidad|precio|metodo_pago|
+-------------+--------+----------+--------------------------------+-----------------+--------+------+-----------+
|1            |1       |2026-10-01|Torta de chocolate              |Pasteles y tortas|1       |7.00  |Efectivo   |
|2            |1       |2026-10-01|Keke de naranja                 |Pasteles y tortas|1       |4.00  |Efectivo   |
|3            |2       |2026-10-01|Té verde                        |Tés e infusiones |4       |14.00 |Efectivo   |
|4            |3       |2026-10-01|Café americano                  |Cafés            |6       |24.00 |Efectivo   |
|5            |4       |2026-10-01|Té verde                        |Tés e infusiones |2       |7.00  |Yape       |
|6            |4       |2026-10-01|Galleta con chispas de chocolate|Postres y du

Paso 7. Eliminar registros duplicados

In [0]:
df = df.dropDuplicates()

print("Registros después de eliminar duplicados:", df.count())

Registros después de eliminar duplicados: 2197


Paso 8. Crear una nueva columna calculada

In [0]:
df_total = df = df.withColumn(
    "importe_total",
    col("precio")
)

In [0]:
from pyspark.sql.functions import sum as spark_sum

df.select(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).show()


+--------------+------------+
|Total_Unidades|Total_Ventas|
+--------------+------------+
|          5107|    20884.60|
+--------------+------------+



Paso 9. Seleccionar columnas necesarias

In [0]:
# Seleccionar información para el análisis
df = df.select(
    "id_fact_venta",
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "importe_total",
    "metodo_pago"
)

df.show(10, truncate=False)

+-------------+--------+----------+----------------+--------------------+--------+------+-------------+-----------+
|id_fact_venta|id_venta|fecha     |producto        |categoria           |cantidad|precio|importe_total|metodo_pago|
+-------------+--------+----------+----------------+--------------------+--------+------+-------------+-----------+
|3            |2       |2026-10-01|Té verde        |Tés e infusiones    |4       |14.00 |14.00        |Efectivo   |
|4            |3       |2026-10-01|Café americano  |Cafés               |6       |24.00 |24.00        |Efectivo   |
|8            |6       |2026-10-01|Café cappuccino |Cafés               |1       |7.00  |7.00         |Yape       |
|16           |10      |2026-10-01|Galletas Oreo   |Galletas y bocaditos|1       |2.00  |2.00         |Efectivo   |
|17           |11      |2026-10-01|Infusión de anís|Tés e infusiones    |2       |6.00  |6.00         |Efectivo   |
|19           |12      |2026-10-01|Infusión de anís|Tés e infusiones    

Paso 10. Agrupar y analizar las ventas

Agrupar las ventas por categoría:

In [0]:
df_ventas_categoria=df.groupBy("categoria").sum("cantidad", "importe_total")
display(df_ventas_categoria)

categoria,sum(cantidad),sum(importe_total)
Tés e infusiones,1761,5615.50
Cafés,1945,8080.00
Galletas y bocaditos,222,755.00
Pasteles y tortas,162,1042.00
Postres y dulces,183,662.00
Panes,157,398.10
Sándwiches y hamburguesas,165,1451.00
Snacks y aperitivos,78,254.00
Desayunos,88,743.00
Bebidas frías,143,531.50


Agrupar las ventas por método de pago:

In [0]:
df_ventas_pago=df.groupBy("metodo_pago").sum("importe_total")
display(df_ventas_pago)

metodo_pago,sum(importe_total)
Efectivo,14484.30
Yape,4600.60
Plin,1450.90
Tarjeta,348.80


Filtrar productos de la categoría Panes:

In [0]:
df_panes_c=df.filter(
    col("categoria") == "Panes"
).select(
    "producto",
    "cantidad",
    "precio",
    "importe_total"
)
display(df_panes_c)

producto,cantidad,precio,importe_total
Pan de queso,2,7.00,7.00
Pan de queso,1,3.50,3.50
Pan francés,1,0.80,0.80
Pan francés,1,0.80,0.80
Croissant,1,4.50,4.50
Pan integral,1,1.50,1.50
Pan de queso,1,3.50,3.50
Croissant,2,9.00,9.00
Pan de queso,1,3.50,3.50
Pan francés,1,0.80,0.80


Analizar las ventas por producto:

In [0]:
df.groupBy("producto") \
    .sum("cantidad", "importe_total") \
    .orderBy(col("sum(importe_total)").desc()) \
    .show(10, truncate=False)

+-------------------------+-------------+------------------+
|producto                 |sum(cantidad)|sum(importe_total)|
+-------------------------+-------------+------------------+
|Café americano           |1825         |7300.00           |
|Infusión de anís         |1040         |3120.00           |
|Té verde                 |591          |2068.50           |
|Jugo surtido             |41           |328.00            |
|Sándwich de pollo        |41           |328.00            |
|Hamburguesa clásica      |32           |320.00            |
|Sándwich de chicharrón   |31           |310.00            |
|Sándwich de pollo y palta|33           |297.00            |
|Cheesecake de fresa      |35           |280.00            |
|Mocaccino                |32           |256.00            |
+-------------------------+-------------+------------------+
only showing top 10 rows


CREACION BASE DE DATOS

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS bd_villarica;


In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bd_villarica.tb_ventas_villarica")

In [0]:

dfVentasGuardadas = spark.table('bd_villarica.tb_ventas_villarica')

display(dfVentasGuardadas)


id_fact_venta,id_venta,fecha,producto,categoria,cantidad,precio,importe_total,metodo_pago
3,2,2026-10-01,Té verde,Tés e infusiones,4,14.00,14.00,Efectivo
4,3,2026-10-01,Café americano,Cafés,6,24.00,24.00,Efectivo
8,6,2026-10-01,Café cappuccino,Cafés,1,7.00,7.00,Yape
16,10,2026-10-01,Galletas Oreo,Galletas y bocaditos,1,2.00,2.00,Efectivo
17,11,2026-10-01,Infusión de anís,Tés e infusiones,2,6.00,6.00,Efectivo
19,12,2026-10-01,Infusión de anís,Tés e infusiones,2,6.00,6.00,Yape
20,13,2026-10-01,Té verde,Tés e infusiones,3,10.50,10.50,Efectivo
38,24,2026-10-01,Pie de limón,Pasteles y tortas,1,6.00,6.00,Efectivo
98,62,2026-10-01,Infusión de anís,Tés e infusiones,4,12.00,12.00,Efectivo
122,74,2026-10-01,Infusión de anís,Tés e infusiones,10,30.00,30.00,Efectivo


GRAFICOS

In [0]:
dfVentasCategoria = df.groupBy("categoria").sum("importe_total")
display(dfVentasCategoria)

categoria,sum(importe_total)
Tés e infusiones,5615.50
Cafés,8080.00
Galletas y bocaditos,755.00
Pasteles y tortas,1042.00
Postres y dulces,662.00
Panes,398.10
Sándwiches y hamburguesas,1451.00
Snacks y aperitivos,254.00
Desayunos,743.00
Bebidas frías,531.50


Databricks visualization. Run in Databricks to view.

In [0]:

dfProductosVendidos = df.groupBy("producto").sum("cantidad")

dfProductosVendidos = dfProductosVendidos.orderBy(
    col("sum(cantidad)").desc()
).limit(10)

display(dfProductosVendidos)


producto,sum(cantidad)
Café americano,1825
Infusión de anís,1040
Té verde,591
Galletas de vainilla,67
Enrollado de hot dog,49
Empanada de pollo,49
Té de manzanilla,48
Té de hierba luisa,45
Jugo surtido,41
Sándwich de pollo,41


Databricks visualization. Run in Databricks to view.

In [0]:

dfVentasPago = df.groupBy("metodo_pago").sum("importe_total")

display(dfVentasPago)


metodo_pago,sum(importe_total)
Efectivo,14484.30
Yape,4600.60
Plin,1450.90
Tarjeta,348.80


Databricks visualization. Run in Databricks to view.

ESCALABILIDAD DEL PROCESAMIENTO

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum

FACTORES = [1, 10, 20, 50]     # 1x = dataset real (2,197 registros)
REPETICIONES = 5               # corridas por volumen
BASE_REGISTROS = df.count()
BASE_UNIDADES = df.agg(spark_sum("cantidad")).first()[0]
BASE_IMPORTE = df.agg(spark_sum("importe_total")).first()[0]
MAX_ID_DET = df.agg(F.max("id_fact_venta")).first()[0]
MAX_ID_VTA = df.agg(F.max("id_venta")).first()[0]

print("Base:", BASE_REGISTROS, "registros |", BASE_UNIDADES, "unidades | S/", BASE_IMPORTE)

Base: 2197 registros | 5107 unidades | S/ 20884.60


In [0]:
def escalar(df_base, factor):
    """Replica el dataset 'factor' veces con ids nuevos (dataset sintético)."""
    if factor == 1:
        return df_base
    k = spark.range(factor).withColumnRenamed("id", "k")
    return (df_base.crossJoin(k)
            .withColumn("id_fact_venta", col("id_fact_venta") + col("k") * F.lit(MAX_ID_DET))
            .withColumn("id_venta", col("id_venta") + col("k") * F.lit(MAX_ID_VTA))
            .drop("k"))

In [0]:
def carga_trabajo(d):
    """Ejecuta las consultas analíticas y fuerza su ejecución con collect()."""
    d.agg(spark_sum("cantidad"), spark_sum("importe_total")).collect()
    d.groupBy("categoria").agg(spark_sum("cantidad"), spark_sum("importe_total")).collect()
    d.groupBy("metodo_pago").agg(spark_sum("importe_total")).collect()
    (d.groupBy("producto").agg(spark_sum("cantidad").alias("u"), spark_sum("importe_total").alias("s"))
       .orderBy(col("s").desc()).limit(10).collect())
    d.groupBy("fecha").agg(spark_sum("importe_total")).collect()

In [0]:
from decimal import Decimal

validacion = []
for f in FACTORES:
    d = escalar(df, f)
    n = d.count()
    u = d.agg(spark_sum("cantidad")).first()[0]
    s = d.agg(spark_sum("importe_total")).first()[0]
    ok = (n == BASE_REGISTROS * f) and (u == BASE_UNIDADES * f) and (s == BASE_IMPORTE * f)
    validacion.append((f, n, int(u), float(s), "CONFORME" if ok else "DIFERENCIA"))

df_val = spark.createDataFrame(validacion, ["factor", "registros", "unidades", "importe_total", "validacion"])
display(df_val)

factor,registros,unidades,importe_total,validacion
1,2197,5107,20884.6,CONFORME
10,21970,51070,208846.0,CONFORME
20,43940,102140,417692.0,CONFORME
50,109850,255350,1044230.0,CONFORME


In [0]:
import statistics, datetime

filas = []
for f in FACTORES:
    d = escalar(df, f)
    n = BASE_REGISTROS * f
    tiempos = []
    for r in range(REPETICIONES + 1):          # +1 = corrida de calentamiento
        t0 = time.perf_counter()
        carga_trabajo(d)
        t = time.perf_counter() - t0
        tiempos.append(t)
        print(f"factor {f}x | corrida {r} | {t:.3f} s" + ("  (calentamiento)" if r == 0 else ""))
    cal, medidos = tiempos[0], tiempos[1:]
    media = statistics.mean(medidos)
    desv = statistics.stdev(medidos) if len(medidos) > 1 else 0.0
    filas.append((f, n, round(cal, 3), round(media, 3), round(desv, 3),
                  round(min(medidos), 3), round(max(medidos), 3), round(n / media, 1)))

df_res = spark.createDataFrame(
    filas,
    ["factor", "registros", "calentamiento_s", "media_s", "desv_s", "min_s", "max_s", "throughput_reg_s"])
display(df_res)

factor 1x | corrida 0 | 2.344 s  (calentamiento)
factor 1x | corrida 1 | 2.190 s
factor 1x | corrida 2 | 2.280 s
factor 1x | corrida 3 | 2.204 s
factor 1x | corrida 4 | 2.281 s
factor 1x | corrida 5 | 2.507 s
factor 10x | corrida 0 | 2.718 s  (calentamiento)
factor 10x | corrida 1 | 2.659 s
factor 10x | corrida 2 | 2.921 s
factor 10x | corrida 3 | 2.589 s
factor 10x | corrida 4 | 2.599 s
factor 10x | corrida 5 | 2.518 s
factor 20x | corrida 0 | 2.717 s  (calentamiento)
factor 20x | corrida 1 | 2.594 s
factor 20x | corrida 2 | 2.803 s
factor 20x | corrida 3 | 2.773 s
factor 20x | corrida 4 | 2.378 s
factor 20x | corrida 5 | 2.345 s
factor 50x | corrida 0 | 2.381 s  (calentamiento)
factor 50x | corrida 1 | 2.585 s
factor 50x | corrida 2 | 2.542 s
factor 50x | corrida 3 | 2.464 s
factor 50x | corrida 4 | 2.322 s
factor 50x | corrida 5 | 2.445 s


factor,registros,calentamiento_s,media_s,desv_s,min_s,max_s,throughput_reg_s
1,2197,2.344,2.292,0.127,2.19,2.507,958.4
10,21970,2.718,2.657,0.156,2.518,2.921,8268.4
20,43940,2.717,2.578,0.214,2.345,2.803,17041.0
50,109850,2.381,2.472,0.101,2.322,2.585,44446.1


In [0]:
esc = []
for f in FACTORES:
    d = escalar(df, f)
    t0 = time.perf_counter()
    d.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"bd_villarica.tb_ventas_escala_{f}x")
    esc.append((f, BASE_REGISTROS * f, round(time.perf_counter() - t0, 3)))

df_esc = spark.createDataFrame(esc, ["factor", "registros", "escritura_s"])
display(df_esc)

factor,registros,escritura_s
1,2197,2.191
10,21970,2.287
20,43940,2.075
50,109850,2.175
